In [ ]:
# hide
import numpy as np
import pyquist as pq

In [ ]:
def osc(freq: float | np.ndarray, T: float, f_s: int = 44100) -> pq.Audio:
    """A time-varying sine oscillator. `freq` gives the frequency (in Hz) at each
    sample, so the pitch can vary over time. We accumulate frequency into phase with
    `np.cumsum` (a running total), then take the sine of the accumulated phase."""
    N = int(T * f_s)
    if np.ndim(freq) == 0:  # a single number -> a constant frequency
        freq = np.full(N, freq)
    if len(freq) != N:
        raise ValueError(f"freq has length {len(freq)}, expected {N} (= T * f_s)")
    theta = np.cumsum(2 * np.pi * freq / f_s)  # accumulated phase, in radians
    return pq.Audio(np.sin(theta).astype(np.float32), f_s)


def fm(f_c: float, f_m: np.ndarray | pq.Audio, D: float, T: float, f_s: int = 44100) -> pq.Audio:
    """General FM synthesis, built on top of `osc`. A carrier at `f_c` Hz has its
    frequency wobbled by a modulator signal `f_m`. `D` is the peak frequency deviation
    in Hertz, so the carrier's instantaneous frequency is `f_c + D * f_m`."""
    if isinstance(f_m, pq.Audio):
        f_m = f_m.as_mono().samples[:, 0]
    return osc(f_c + D * f_m, T, f_s)


def fm_classic(f_c: float, f_m: float, I: float, T: float, f_s: int = 44100) -> pq.Audio:
    """Classic FM synthesis, built on top of two calls to `osc`. Carrier at `f_c` Hz
    and sinusoidal modulation at `f_m` Hz; `I = D / f_m` is the index of modulation."""
    D = I * f_m
    return osc(f_c + D * osc(f_m, T), T, f_s)
    #return fm(f_c, osc(f_m, T), I * f_m, T)   # equivalent


T = 4.0
f_c = 440.0
f_m = 4.0
I = 20.0

basic_osc = osc(f_c, T) # basic sinusoid at f_c
fm_osc = fm(f_c, osc(f_m, T), I * f_m, T)  # classic FM with parameters f_c/f_m/I
#fm_osc = fm_classic(f_c, f_m, I, T)   # equivalent

pq.play(basic_osc)
pq.plot_spec(basic_osc)
pq.play(fm_osc)
pq.plot_spec(fm_osc)